In [3]:
import numpy as np
import pandas as pd

In [27]:
import pandas as pd

df_list = []

for i in range(2301, 2313):
    temp = pd.read_csv(f"../../3조 공유폴더/2023 강남구 따릉이 이용정보/{i}_강남구_따릉이_이용정보.csv")
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

df['대여일시'] = pd.to_datetime(df['대여일시'])
df['반납일시'] = pd.to_datetime(df['반납일시'])

In [ ]:
rent_event = df[['대여 대여소번호','대여일시']].copy()
rent_event['change'] = -1
rent_event = rent_event.rename(columns={'대여 대여소번호':'station','대여일시':'time'})

return_event = df[['반납대여소번호','반납일시']].copy()
return_event['change'] = +1
return_event = return_event.rename(columns={'반납대여소번호':'station','반납일시':'time'})

event = pd.concat([rent_event, return_event])
event['time'] = pd.to_datetime(event['time'])

event = event.sort_values('time')

event['time'] = event['time'].dt.floor('h')

hourly_change = (event.groupby(['station','time'])['change'].sum().reset_index())

time_range = pd.date_range(start='2023-01-01', end='2024-12-31 23:00', freq='h')

stations = event['station'].unique()

grid = pd.MultiIndex.from_product([stations, time_range], names=['station','time']).to_frame(index=False)

df_hour = grid.merge(hourly_change, how='left')
df_hour['change'] = df_hour['change'].fillna(0)

df_hour['bike_count'] = (df_hour.sort_values('time').groupby('station')['change'].cumsum())

df_hour

,station,time,change,bike_count
0,2347.0,2023-01-01 00:00:00,-1.0,-1.0
1,2347.0,2023-01-01 01:00:00,0.0,-1.0
2,2347.0,2023-01-01 02:00:00,0.0,-1.0
3,2347.0,2023-01-01 03:00:00,0.0,-1.0
4,2347.0,2023-01-01 04:00:00,0.0,-1.0
...,...,...,...,...
2982475,4930.0,2024-12-31 19:00:00,0.0,-31.0
2982476,4930.0,2024-12-31 20:00:00,0.0,-31.0
2982477,4930.0,2024-12-31 21:00:00,0.0,-31.0
2982478,4930.0,2024-12-31 22:00:00,0.0,-31.0
